<a href="https://colab.research.google.com/github/Vishal-3600/100-Days-of-ML/blob/main/CustomerChurnPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Mount Drive (run this FIRST — files will persist here)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


In [ ]:
# Cell 2: Create entire project structure in one go
import os

BASE = "/content/drive/MyDrive/churn_prediction"

folders = [
    "data/raw",
    "data/processed",
    "data/external",
    "notebooks",
    "src",
    "models",
    "api",
    "tests",
    "reports/figures",
    "logs",
]

for folder in folders:
    path = os.path.join(BASE, folder)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

print("\nAll directories created!")

Created: /content/drive/MyDrive/churn_prediction/data/raw
Created: /content/drive/MyDrive/churn_prediction/data/processed
Created: /content/drive/MyDrive/churn_prediction/data/external
Created: /content/drive/MyDrive/churn_prediction/notebooks
Created: /content/drive/MyDrive/churn_prediction/src
Created: /content/drive/MyDrive/churn_prediction/models
Created: /content/drive/MyDrive/churn_prediction/api
Created: /content/drive/MyDrive/churn_prediction/tests
Created: /content/drive/MyDrive/churn_prediction/reports/figures
Created: /content/drive/MyDrive/churn_prediction/logs

All directories created!


In [ ]:
# Cell 3: Create requirements.txt
requirements = """pandas>=2.1.0
numpy>=1.24.3
scikit-learn>=1.3.0
xgboost>=2.0.0
imbalanced-learn>=0.11.0
matplotlib>=3.7.2
seaborn>=0.12.2
mlflow>=2.8.0
joblib>=1.3.2
fastapi>=0.104.0
uvicorn>=0.24.0
pydantic>=2.4.2
python-dotenv>=1.0.0
pytest>=7.4.3
shap>=0.43.0
PyYAML>=6.0.1
"""

with open(f"{BASE}/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [ ]:
# Cell 4: Create config.yaml (central config — no hardcoding!)
config = """
project:
  name: "Customer Churn Prediction"
  version: "1.0.0"

data:
  raw_path: "data/raw/telco_churn.csv"
  processed_path: "data/processed/churn_clean.csv"
  test_size: 0.2
  random_state: 42

features:
  target_column: "Churn"
  drop_columns: ["customerID"]
  categorical_cols:
    - "gender"
    - "Partner"
    - "Dependents"
    - "PhoneService"
    - "MultipleLines"
    - "InternetService"
    - "OnlineSecurity"
    - "Contract"
    - "PaymentMethod"
  numerical_cols:
    - "tenure"
    - "MonthlyCharges"
    - "TotalCharges"

model:
  algorithm: "GradientBoosting"
  n_estimators: 200
  learning_rate: 0.05
  max_depth: 4
  save_path: "models/churn_model.pkl"
  scaler_path: "models/scaler.pkl"

mlflow:
  experiment_name: "churn_prediction"
  tracking_uri: "mlruns/"

api:
  host: "0.0.0.0"
  port: 8000
"""

with open(f"{BASE}/config.yaml", "w") as f:
    f.write(config)

print("config.yaml created")

config.yaml created


In [ ]:
# Cell 5: Create src/preprocess.py
preprocess_code = '''
import pandas as pd
import numpy as np
import yaml
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def load_config(path="config.yaml"):
    with open(path) as f:
        return yaml.safe_load(f)

def load_data(path):
    df = pd.read_csv(path)
    print(f"Loaded data: {df.shape}")
    return df

def clean_data(df, config):
    # Fix TotalCharges
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

    # Drop unwanted columns
    df.drop(columns=config["features"]["drop_columns"], inplace=True, errors="ignore")

    # Encode target
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    return df

def build_preprocessor(config):
    num_cols = config["features"]["numerical_cols"]
    cat_cols = config["features"]["categorical_cols"]

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    return preprocessor

def save_processed(df, path):
    df.to_csv(path, index=False)
    print(f"Saved processed data to {path}")

if __name__ == "__main__":
    config = load_config()
    df = load_data(config["data"]["raw_path"])
    df = clean_data(df, config)
    save_processed(df, config["data"]["processed_path"])
'''

with open(f"{BASE}/src/preprocess.py", "w") as f:
    f.write(preprocess_code)

print("src/preprocess.py created")

src/preprocess.py created


In [ ]:
# Cell 6: Create src/train.py
train_code = '''
import pandas as pd
import numpy as np
import yaml
import joblib
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from preprocess import load_config, build_preprocessor

def train(config_path="config.yaml"):
    config = load_config(config_path)

    # Load processed data
    df = pd.read_csv(config["data"]["processed_path"])
    target = config["features"]["target_column"]

    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config["data"]["test_size"],
        stratify=y,
        random_state=config["data"]["random_state"]
    )

    preprocessor = build_preprocessor(config)

    mlflow.set_experiment(config["mlflow"]["experiment_name"])

    with mlflow.start_run(run_name="gradient_boosting_v1"):
        pipeline = ImbPipeline([
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", GradientBoostingClassifier(
                n_estimators=config["model"]["n_estimators"],
                learning_rate=config["model"]["learning_rate"],
                max_depth=config["model"]["max_depth"],
                random_state=42
            ))
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)

        print(classification_report(y_test, y_pred))
        print(f"ROC-AUC: {auc:.4f}")

        # Log to MLflow
        mlflow.log_params(config["model"])
        mlflow.log_metric("roc_auc", auc)
        mlflow.sklearn.log_model(pipeline, "churn_model")

        # Save locally
        joblib.dump(pipeline, config["model"]["save_path"])
        print(f"Model saved to {config['model']['save_path']}")

    return pipeline

if __name__ == "__main__":
    train()
'''

with open(f"{BASE}/src/train.py", "w") as f:
    f.write(train_code)

print("src/train.py created")

src/train.py created


In [ ]:
# Cell 7: Create src/evaluate.py
evaluate_code = '''
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay
)
from preprocess import load_config

def evaluate(config_path="config.yaml"):
    config = load_config(config_path)

    df = pd.read_csv(config["data"]["processed_path"])
    target = config["features"]["target_column"]
    X = df.drop(columns=[target])
    y = df[target]

    model = joblib.load(config["model"]["save_path"])
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    print("=== Classification Report ===")
    print(classification_report(y, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y, y_prob):.4f}")

    # Confusion matrix plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[0])
    RocCurveDisplay.from_predictions(y, y_prob, ax=axes[1])
    plt.tight_layout()
    plt.savefig("reports/figures/evaluation.png", dpi=150)
    plt.show()
    print("Plot saved to reports/figures/evaluation.png")

if __name__ == "__main__":
    evaluate()
'''

with open(f"{BASE}/src/evaluate.py", "w") as f:
    f.write(evaluate_code)

print("src/evaluate.py created")

src/evaluate.py created


In [ ]:
# Cell 8: Create api/app.py
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import yaml
import os

app = FastAPI(
    title="Churn Prediction API",
    description="Predict customer churn probability",
    version="1.0.0"
)

# Load model on startup
model = None

@app.on_event("startup")
def load_model():
    global model
    with open("config.yaml") as f:
        config = yaml.safe_load(f)
    model = joblib.load(config["model"]["save_path"])
    print("Model loaded successfully")

class CustomerFeatures(BaseModel):
    tenure: int
    MonthlyCharges: float
    TotalCharges: float
    gender: str
    Partner: str
    Dependents: str
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    Contract: str
    PaymentMethod: str

class PredictionResponse(BaseModel):
    churn: bool
    churn_probability: float
    risk_level: str

@app.post("/predict", response_model=PredictionResponse)
def predict(data: CustomerFeatures):
    try:
        input_df = pd.DataFrame([data.dict()])
        probability = float(model.predict_proba(input_df)[0][1])
        prediction = bool(probability >= 0.5)

        if probability >= 0.7:
            risk = "High"
        elif probability >= 0.4:
            risk = "Medium"
        else:
            risk = "Low"

        return PredictionResponse(
            churn=prediction,
            churn_probability=round(probability, 4),
            risk_level=risk
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.get("/")
def root():
    return {"message": "Churn Prediction API is running"}
'''

with open(f"{BASE}/api/app.py", "w") as f:
    f.write(api_code)

print("api/app.py created")

api/app.py created


In [ ]:
# Cell 9: Create tests/test_model.py
test_code = '''
import pytest
import pandas as pd
import numpy as np
import joblib
import sys
sys.path.insert(0, "../src")

def test_model_loads():
    model = joblib.load("../models/churn_model.pkl")
    assert model is not None

def test_model_predicts():
    model = joblib.load("../models/churn_model.pkl")
    sample = pd.DataFrame([{
        "tenure": 12,
        "MonthlyCharges": 65.5,
        "TotalCharges": 786.0,
        "gender": "Male",
        "Partner": "Yes",
        "Dependents": "No",
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "Fiber optic",
        "OnlineSecurity": "No",
        "Contract": "Month-to-month",
        "PaymentMethod": "Electronic check"
    }])
    pred = model.predict(sample)
    assert pred[0] in [0, 1]

def test_probability_range():
    model = joblib.load("../models/churn_model.pkl")
    sample = pd.DataFrame([{
        "tenure": 24, "MonthlyCharges": 50.0, "TotalCharges": 1200.0,
        "gender": "Female", "Partner": "No", "Dependents": "No",
        "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": "DSL", "OnlineSecurity": "Yes",
        "Contract": "One year", "PaymentMethod": "Bank transfer (automatic)"
    }])
    prob = model.predict_proba(sample)[0][1]
    assert 0.0 <= prob <= 1.0
'''

with open(f"{BASE}/tests/test_model.py", "w") as f:
    f.write(test_code)

print("tests/test_model.py created")

tests/test_model.py created


In [ ]:
# Cell 10: Create Dockerfile
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000
CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open(f"{BASE}/Dockerfile", "w") as f:
    f.write(dockerfile)

# Create .gitignore
gitignore = """# Data (never commit raw data)
data/raw/
data/processed/

# Models (large binary files)
models/*.pkl
models/*.joblib

# Colab/Jupyter
.ipynb_checkpoints/
*.pyc
__pycache__/

# Environment
.env
*.env

# MLflow
mlruns/
mlartifacts/

# Logs
logs/
*.log
"""

with open(f"{BASE}/.gitignore", "w") as f:
    f.write(gitignore)

print("Dockerfile and .gitignore created")

Dockerfile and .gitignore created


In [ ]:
# Cell 11: Print the full directory tree to verify everything
import os

def print_tree(start_path, indent=0):
    for item in sorted(os.listdir(start_path)):
        if item.startswith('.') and item != '.gitignore':
            continue
        full_path = os.path.join(start_path, item)
        prefix = "    " * indent + ("📁 " if os.path.isdir(full_path) else "📄 ")
        print(prefix + item)
        if os.path.isdir(full_path):
            print_tree(full_path, indent + 1)

print(f"Project structure at: {BASE}\n")
print_tree(BASE)

Project structure at: /content/drive/MyDrive/churn_prediction

📄 .gitignore
📄 Dockerfile
📁 api
    📄 app.py
📄 config.yaml
📁 data
    📁 external
    📁 processed
    📁 raw
📁 logs
📁 models
📁 notebooks
📁 reports
    📁 figures
📄 requirements.txt
📁 src
    📄 evaluate.py
    📄 preprocess.py
    📄 train.py
📁 tests
    📄 test_model.py


In [ ]:
import pandas as pd
import numpy as np
import sklearn
import xgboost
import mlflow
import fastapi
import shap

print("All libraries imported successfully 🚀")

All libraries imported successfully 🚀


In [ ]:
import subprocess

BASE = "/content/drive/MyDrive/churn_prediction"

result = subprocess.run(
    ["pip", "install", "-r", f"{BASE}/requirements.txt"],
    text=True
)

print("Installation complete!" if result.returncode == 0 else result.stderr)

Installation complete!


In [ ]:
# Cell 13: Set working directory so imports work correctly
import os, sys

os.chdir(BASE)
sys.path.insert(0, f"{BASE}/src")

print(f"Working directory: {os.getcwd()}")
print("Project is ready. Next step: download the dataset and place it in data/raw/")

Working directory: /content/drive/MyDrive/churn_prediction
Project is ready. Next step: download the dataset and place it in data/raw/


In [ ]:
# Cell 14: Download the Telco dataset directly into data/raw/
import urllib.request

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
save_path = f"{BASE}/data/raw/telco_churn.csv"

urllib.request.urlretrieve(url, save_path)
print(f"Dataset downloaded to: {save_path}")

# Quick check
import pandas as pd
df = pd.read_csv(save_path)
print(f"Shape: {df.shape}")
print(df['Churn'].value_counts())

Dataset downloaded to: /content/drive/MyDrive/churn_prediction/data/raw/telco_churn.csv
Shape: (7043, 21)
Churn
No     5174
Yes    1869
Name: count, dtype: int64
